# Multivariate LSTM Ensemble (E2) — Arctic Sea Ice Extent Forecasting

A 10-member ensemble of multivariate LSTMs (ERA5 atmospheric variables + a 90-day lookback),
forecasting pan-Arctic sea-ice extent one day ahead. This is the **multivariate** counterpart to
the univariate `09_lstm_ensemble`, and the ensemble deliverable of experiment **E2**.

**Methodology (aligned with the project conventions):**
- **Three-way temporal split** — train 1989–2014, validation 2015–2019 (early stopping /
  model selection), test 2020–2023 touched once at final evaluation.
- Evaluated against **persistence** and **climatology** baselines using the shared
  `src.evaluation_utils` engine (RMSE / MAE / MAPE, skill scores, ACC, PICP / MPIW, Diebold–Mariano).
- Results logged to `results/model_comparison.csv` (`scale="daily"`) so `07_model_comparison`
  picks them up automatically. Figures saved to `results/figures/`.

> Status: **pending GPU run** — outputs cleared; rerun end-to-end to regenerate metrics and figures.

In [ ]:
import sys; sys.path.append("..")

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error

from src.data_utils import load_data
from src.evaluation_utils import (
    ClimatologyModel, compute_all_metrics, compute_rmse,
    diebold_mariano_test, holm_bonferroni, picp, mpiw, log_model_results,
)

## Dataset Class

Using the same multivariate dataset class from the previous notebook.

In [ ]:
class MultivariateArcticDataset(torch.utils.data.Dataset):

    def __init__(self, data, sequence_length=30, forecast_horizon=1, features=None, target='extent_mkm2', scaler=None, lag_features=None, add_cyclical_time=False):
        self.data = data.sort_values('date').reset_index(drop=True)
        self.sequence_length = sequence_length
        self.forecast_horizon = forecast_horizon
        self.target = target
        
        if features is None:
            self.features = ['extent_mkm2']
        else:
            self.features = features.copy()
        
        if add_cyclical_time:
            day_of_year = pd.to_datetime(self.data['date']).dt.dayofyear
            self.data['day_of_year_sin'] = np.sin(2 * np.pi * day_of_year / 365.25)
            self.data['day_of_year_cos'] = np.cos(2 * np.pi * day_of_year / 365.25)
            self.features.extend(['day_of_year_sin', 'day_of_year_cos'])
        
        if lag_features is not None:
            for column, lags in lag_features.items():
                for lag_days in lags:
                    lagged_column_name = f"{column}_lag{lag_days}"
                    self.data[lagged_column_name] = self.data[column].shift(lag_days)
                    self.features.append(lagged_column_name)
            
            self.data = self.data.dropna().reset_index(drop=True)
        
        self.data_values = self.data[self.features].values.astype(np.float32)
        
        self.target_idx = self.features.index(self.target)
        
        if scaler is None:
            self.mean = self.data_values.mean(axis=0, keepdims=True)
            self.std = self.data_values.std(axis=0, keepdims=True)
            self.std = np.where(self.std == 0, 1.0, self.std)
        else:
            self.mean, self.std = scaler
        
        self.data_normalized = (self.data_values - self.mean) / self.std
    
    def __len__(self):
        return len(self.data_normalized) - self.sequence_length - self.forecast_horizon + 1
    
    def __getitem__(self, idx):
        X = self.data_normalized[idx:idx + self.sequence_length]
        
        y = self.data_normalized[idx + self.sequence_length + self.forecast_horizon - 1][self.target_idx]
        
        X = torch.tensor(X, dtype=torch.float32)
        y = torch.tensor(y, dtype=torch.float32)
        
        return X, y

## Model Architecture

In [ ]:
class IceExtentLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, output_size=1, dropout=0.2):
        super(IceExtentLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        h_0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c_0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        
        out, (h_n, c_n) = self.lstm(x, (h_0, c_0))
        
        out = self.fc(out[:, -1, :])
        
        return out

## Load Data and Prepare Features

In [ ]:
# Three-way temporal split — matches the shared project convention (see 04_basic_lstm and
# 09_lstm_ensemble). The validation era drives early stopping / model selection; the test era is
# touched only once, at final evaluation. Previously this notebook used the *test* set as the
# early-stopping signal, which leaked test information into model selection.
#   train 1989–2014  |  val 2015–2019  |  test 2020–2023
train_data = load_data(regions='pan_arctic', years=range(1989, 2015))
val_data   = load_data(regions='pan_arctic', years=range(2015, 2020))
test_data  = load_data(regions='pan_arctic', years=range(2020, 2024))

print(f"Training data shape:   {train_data.shape}")
print(f"Validation data shape: {val_data.shape}")
print(f"Test data shape:       {test_data.shape}")

In [ ]:
# Define features: atmospheric aggregated variables (non-lagged)
features = [
    'extent_mkm2',
    't2m_mean',
    't2m_std',
    'msl_mean',
    'msl_std',
    'wind_speed_mean',
    'wind_speed_std',
]

print(f"Features: {features}")
print(f"Number of features: {len(features)}")

## Create Datasets with 90-day Lookback

In [ ]:
# Create datasets with a 90-day sequence length. The scaler is fit on the TRAINING set only and
# reused for val/test, so no future statistics leak into normalization.
train_dataset = MultivariateArcticDataset(
    train_data, sequence_length=90, forecast_horizon=1, features=features, target='extent_mkm2',
)

val_dataset = MultivariateArcticDataset(
    val_data, sequence_length=90, forecast_horizon=1, features=features, target='extent_mkm2',
    scaler=(train_dataset.mean, train_dataset.std),
)

test_dataset = MultivariateArcticDataset(
    test_data, sequence_length=90, forecast_horizon=1, features=features, target='extent_mkm2',
    scaler=(train_dataset.mean, train_dataset.std),
)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = torch.utils.data.DataLoader(val_dataset,   batch_size=32, shuffle=False)
test_loader  = torch.utils.data.DataLoader(test_dataset,  batch_size=32, shuffle=False)

print(f"Training samples:   {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples:       {len(test_dataset)}")

for X_batch, y_batch in train_loader:
    print(f"\nBatch X shape: {X_batch.shape}")
    print(f"Batch y shape: {y_batch.shape}")
    break

## Train Ensemble of 10 Models

Each model is initialized with different random weights to capture different aspects of the data.

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Ensemble configuration
num_ensemble_models = 10
num_epochs = 150
patience = 15

ensemble_models = []
ensemble_train_losses = []
ensemble_val_losses = []
ensemble_best_losses = []

print("=" * 70)
print(f"TRAINING ENSEMBLE OF {num_ensemble_models} MULTIVARIATE LSTM MODELS")
print(f"Lookback window: 90 days")
print(f"Features: {len(features)}")
print("=" * 70)

for model_idx in range(num_ensemble_models):
    print(f"\n{'='*70}")
    print(f"Training Model {model_idx + 1}/{num_ensemble_models}")
    print(f"{'='*70}")

    # Set random seed for reproducibility while ensuring different initializations
    seed = 42 + model_idx
    torch.manual_seed(seed)
    np.random.seed(seed)

    # Initialize fresh model with different random weights
    model = IceExtentLSTM(
        input_size=len(features),
        hidden_size=80,
        num_layers=2,
        output_size=1,
        dropout=0.2
    )
    model = model.to(device)

    # Fresh optimizer and scheduler for each model
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

    best_val_loss = float('inf')
    patience_counter = 0
    train_losses = []
    val_losses = []

    # Training loop for this model
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            predictions = model(X_batch)
            loss = criterion(predictions.squeeze(), y_batch)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)

        # Validation on the VALIDATION era (2015-2019) — this is the early-stopping /
        # model-selection signal. The test era is never seen during training.
        model.eval()
        val_loss = 0.0

        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)

                predictions = model(X_batch)
                loss = criterion(predictions.squeeze(), y_batch)
                val_loss += loss.item()

        avg_val_loss = val_loss / len(val_loader)

        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)

        scheduler.step(avg_val_loss)

        # Print progress every 10 epochs for ensemble
        if (epoch + 1) % 10 == 0:
            print(f'  Epoch {epoch+1:3d}/{num_epochs} | Train: {avg_train_loss:.6f} | Val: {avg_val_loss:.6f}')

        # Early stopping
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"  Early stopping at epoch {epoch+1}")
                break

    # Load best weights for this model
    model.load_state_dict(best_model_state)

    # Store model and training history
    ensemble_models.append(model)
    ensemble_train_losses.append(train_losses)
    ensemble_val_losses.append(val_losses)
    ensemble_best_losses.append(best_val_loss)

    print(f"  ✓ Model {model_idx + 1} complete! Best val loss: {best_val_loss:.6f}")

print(f"\n{'='*70}")
print("ENSEMBLE TRAINING COMPLETE!")
print(f"{'='*70}")
print(f"Best validation losses across {num_ensemble_models} models:")
for i, loss in enumerate(ensemble_best_losses):
    print(f"  Model {i+1}: {loss:.6f}")
print(f"\nMean: {np.mean(ensemble_best_losses):.6f} ± {np.std(ensemble_best_losses):.6f}")
print(f"Range: [{np.min(ensemble_best_losses):.6f}, {np.max(ensemble_best_losses):.6f}]")

## Generate Ensemble Predictions

In [ ]:
print("Generating ensemble predictions...")

all_predictions = []

for model_idx, model in enumerate(ensemble_models):
    model.eval()
    predictions = []
    
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            pred = model(X_batch)
            predictions.append(pred.cpu().numpy())
    
    predictions = np.concatenate(predictions, axis=0).squeeze()
    all_predictions.append(predictions)

# Convert to numpy array: shape (num_models, num_samples)
all_predictions = np.array(all_predictions)

# Compute ensemble statistics
ensemble_mean = all_predictions.mean(axis=0)
ensemble_std = all_predictions.std(axis=0)
ensemble_median = np.median(all_predictions, axis=0)

# Compute confidence intervals
ensemble_q25 = np.percentile(all_predictions, 25, axis=0)
ensemble_q75 = np.percentile(all_predictions, 75, axis=0)
ensemble_q05 = np.percentile(all_predictions, 5, axis=0)
ensemble_q95 = np.percentile(all_predictions, 95, axis=0)

print(f"✓ Generated predictions from {len(ensemble_models)} models")
print(f"  Predictions shape: {all_predictions.shape}")
print(f"  Mean ensemble std: {ensemble_std.mean():.6f}")
print(f"  Max ensemble std: {ensemble_std.max():.6f}")
print(f"  Min ensemble std: {ensemble_std.min():.6f}")

In [ ]:
# Get actual test values for comparison
actual_values = []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        actual_values.append(y_batch.cpu().numpy())
actual_values = np.concatenate(actual_values, axis=0)

# Denormalize predictions back to original scale (Mkm²)
mean_val = train_dataset.mean[0, train_dataset.target_idx]
std_val = train_dataset.std[0, train_dataset.target_idx]

ensemble_mean_denorm = ensemble_mean * std_val + mean_val
ensemble_std_denorm = ensemble_std * std_val
ensemble_q05_denorm = ensemble_q05 * std_val + mean_val
ensemble_q95_denorm = ensemble_q95 * std_val + mean_val
ensemble_q25_denorm = ensemble_q25 * std_val + mean_val
ensemble_q75_denorm = ensemble_q75 * std_val + mean_val
actual_values_denorm = actual_values * std_val + mean_val

# Create time index for plotting
test_dates = test_data.sort_values('date')['date'].values[90:]  # Account for sequence length
test_dates = test_dates[:len(actual_values_denorm)]  # Match actual predictions length

print(f"Test dates range: {test_dates[0]} to {test_dates[-1]}")
print(f"Number of predictions: {len(ensemble_mean_denorm)}")

## Visualize Ensemble Predictions

In [ ]:
# Visualize ensemble predictions with uncertainty bands

# Plot 3: Prediction uncertainty over time
fig, ax = plt.subplots(1, 1, figsize=(16, 6))

ax.plot(test_dates, ensemble_std_denorm, 'purple', linewidth=2)
ax.fill_between(test_dates, 0, ensemble_std_denorm, alpha=0.3, color='purple')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Prediction Std Dev (Mkm²)', fontsize=12)
ax.set_title('Ensemble Uncertainty Over Time', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig("../results/figures/09b_multivariate_ensemble_uncertainty.png", dpi=120, bbox_inches="tight")
plt.show()

print(f"\nUncertainty Statistics:")
print(f"  Mean uncertainty (std): {ensemble_std_denorm.mean():.4f} Mkm²")
print(f"  Median uncertainty: {np.median(ensemble_std_denorm):.4f} Mkm²")
print(f"  Max uncertainty: {ensemble_std_denorm.max():.4f} Mkm²")
print(f"  Min uncertainty: {ensemble_std_denorm.min():.4f} Mkm²")


## Evaluate Performance

In [ ]:
# ============================================================================
# Standardized evaluation via src.evaluation_utils (baselines + skill + significance)
# ============================================================================

# --- Baselines on the test era (denormalized, Mkm²) --------------------------------------
# Persistence: forecast(t) = observed extent at t-1 (horizon = 1). The aligned target rows begin
# at `start = sequence_length + forecast_horizon - 1`, so persistence is the extent one row
# earlier. Climatology: day-of-year mean fit on the TRAIN era only.
start = test_dataset.sequence_length + test_dataset.forecast_horizon - 1
extent_series = test_dataset.data['extent_mkm2'].values  # physical units, full (sorted) test series
n = len(actual_values_denorm)
persistence = extent_series[start - 1 : start - 1 + n]

clim = ClimatologyModel()
clim.fit(dates=train_data['date'], values=train_data['extent_mkm2'])
climatology = clim.predict(pd.Series(pd.to_datetime(test_dates)))

# --- Standardized metrics -----------------------------------------------------------------
metrics = compute_all_metrics(
    y_true=actual_values_denorm,
    y_pred=ensemble_mean_denorm,
    y_baseline_persistence=persistence,
    y_baseline_climatology=climatology,
    climatology=climatology,
)

print("=" * 70)
print("MULTIVARIATE ENSEMBLE — STANDARDIZED METRICS (test era 2020–2023, Mkm²)")
print("=" * 70)
print(f"  RMSE:                      {metrics['rmse']:.4f} Mkm²")
print(f"  MAE:                       {metrics['mae']:.4f} Mkm²")
print(f"  MAPE:                      {metrics['mape']:.4f} %")
print(f"  Skill vs persistence:      {metrics['skill_score_persistence']:.2%}")
print(f"  Skill vs climatology:      {metrics['skill_score_climatology']:.2%}")
print(f"  Anomaly correlation (ACC): {metrics['anomaly_correlation']:.4f}")

# --- Interval calibration on the 90% ensemble interval ------------------------------------
coverage_90 = picp(actual_values_denorm, ensemble_q05_denorm, ensemble_q95_denorm)
width_90 = mpiw(ensemble_q05_denorm, ensemble_q95_denorm)
print(f"\n  90% interval — nominal 0.90, observed PICP {coverage_90:.3f}, mean width {width_90:.4f} Mkm²")

# --- Per-member spread + Diebold–Mariano significance (Holm–Bonferroni corrected) ---------
individual_rmses = [compute_rmse(actual_values_denorm, all_predictions[i] * std_val + mean_val)
                    for i in range(len(ensemble_models))]
print(f"  Per-member RMSE: mean {np.mean(individual_rmses):.4f}, std {np.std(individual_rmses):.4f} Mkm²")

# Compare against a representative single member (closest to the median member RMSE) so the
# "does ensembling help?" test reflects a typical run rather than a cherry-picked seed.
typical = int(np.argmin(np.abs(np.array(individual_rmses) - np.median(individual_rmses))))
typical_pred = all_predictions[typical] * std_val + mean_val

p_values = {}
_, p_values["ensemble vs persistence"] = diebold_mariano_test(actual_values_denorm, ensemble_mean_denorm, persistence)
_, p_values["ensemble vs climatology"] = diebold_mariano_test(actual_values_denorm, ensemble_mean_denorm, climatology)
_, p_values[f"ensemble vs typical member (seed {42 + typical})"] = diebold_mariano_test(
    actual_values_denorm, ensemble_mean_denorm, typical_pred)

print("\nDiebold–Mariano tests (negative stat ⇒ ensemble better; Holm–Bonferroni across the family):")
print(holm_bonferroni(p_values).to_string(index=False))

# --- Log to the single shared comparison table --------------------------------------------
log_model_results(
    model_name="LSTM_Ensemble10_Multivariate",
    metrics=metrics,
    scale="daily",
    metadata={
        "architecture": "2-layer LSTM, 80 hidden, dropout=0.2",
        "input_features": ", ".join(features),
        "lookback_days": 90,
        "n_seeds": len(ensemble_models),
        "member_rmse_mean": f"{np.mean(individual_rmses):.4f}",
        "member_rmse_std": f"{np.std(individual_rmses):.4f}",
        "picp_90": f"{coverage_90:.3f}",
        "mpiw_90": f"{width_90:.4f}",
        "train_period": "1989-2014", "val_period": "2015-2019", "test_period": "2020-2023",
    },
    output_file="../results/model_comparison.csv",
)

## Random Week Visualization

Detailed view of predictions for randomly selected weeks.

In [ ]:
# Visualize random week-long predictions with model disagreement
np.random.seed(42)  # For reproducibility

# Parameters
num_weeks = 6
days_per_week = 7

# Randomly select week starting indices (ensure we have full weeks)
max_start_idx = len(test_dates) - days_per_week
random_week_starts = np.random.choice(max_start_idx, size=num_weeks, replace=False)
random_week_starts = np.sort(random_week_starts)

print("=" * 70)
print("RANDOMLY SELECTED WEEKS FOR DETAILED ANALYSIS")
print("=" * 70)

for i, start_idx in enumerate(random_week_starts):
    end_idx = start_idx + days_per_week
    week_dates = test_dates[start_idx:end_idx]
    print(f"Week {i+1}: {pd.to_datetime(week_dates[0]).strftime('%Y-%m-%d')} to {pd.to_datetime(week_dates[-1]).strftime('%Y-%m-%d')}")

# Create visualization
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

for week_idx, start_idx in enumerate(random_week_starts):
    ax = axes[week_idx]
    end_idx = start_idx + days_per_week
    
    # Extract data for this week
    week_indices = np.arange(start_idx, end_idx)
    week_dates_subset = test_dates[start_idx:end_idx]
    week_dates_dt = pd.to_datetime(week_dates_subset)
    
    # Get predictions from all models (denormalized)
    week_predictions_all_models = []
    for model_idx in range(len(ensemble_models)):
        pred_denorm = all_predictions[model_idx] * std_val + mean_val
        week_predictions_all_models.append(pred_denorm[start_idx:end_idx])
    
    # Get ensemble statistics for this week
    week_mean = ensemble_mean_denorm[start_idx:end_idx]
    week_q05 = ensemble_q05_denorm[start_idx:end_idx]
    week_q95 = ensemble_q95_denorm[start_idx:end_idx]
    week_q25 = ensemble_q25_denorm[start_idx:end_idx]
    week_q75 = ensemble_q75_denorm[start_idx:end_idx]
    week_actual = actual_values_denorm[start_idx:end_idx]
    
    # Plot confidence intervals first (background)
    ax.fill_between(week_dates_dt, week_q05, week_q95, 
                     alpha=0.2, color='blue', label='90% CI')
    ax.fill_between(week_dates_dt, week_q25, week_q75, 
                     alpha=0.3, color='blue', label='50% CI')
    
    # Plot individual model predictions
    for model_idx in range(len(ensemble_models)):
        ax.plot(week_dates_dt, week_predictions_all_models[model_idx], 
                'o-', alpha=0.4, linewidth=1, markersize=4, color='gray',
                label='Individual Models' if model_idx == 0 else '')
    
    # Plot ensemble mean
    ax.plot(week_dates_dt, week_mean, 'b-', linewidth=3, 
            marker='s', markersize=7, label='Ensemble Mean', zorder=5)
    
    # Plot actual values
    ax.plot(week_dates_dt, week_actual, 'r-', linewidth=3, 
            marker='o', markersize=8, label='Actual', zorder=10)
    
    # Formatting
    ax.set_xlabel('Date', fontsize=11)
    ax.set_ylabel('Ice Extent (Mkm²)', fontsize=11)
    
    week_start_str = pd.to_datetime(week_dates_subset[0]).strftime('%Y-%m-%d')
    week_end_str = pd.to_datetime(week_dates_subset[-1]).strftime('%Y-%m-%d')
    ax.set_title(f'Week {week_idx + 1}: {week_start_str} to {week_end_str}', 
                 fontsize=12, fontweight='bold')
    
    # Only show legend on first subplot
    if week_idx == 0:
        ax.legend(loc='best', fontsize=9)
    
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=45, labelsize=9)
    
    # Calculate and display statistics for this week
    week_mae = mean_absolute_error(week_actual, week_mean)
    week_uncertainty = np.mean(week_q95 - week_q05)
    
    # Add text box with statistics
    textstr = f'MAE: {week_mae:.4f} Mkm²\nUncert: {week_uncertainty:.4f} Mkm²'
    props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
    ax.text(0.02, 0.98, textstr, transform=ax.transAxes, fontsize=9,
            verticalalignment='top', bbox=props)

plt.suptitle('Multivariate Ensemble Predictions for Randomly Selected Weeks', 
             fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig("../results/figures/09b_multivariate_ensemble_weekly.png", dpi=120, bbox_inches="tight")
plt.show()

# Print detailed statistics for each week
print(f"\n{'='*70}")
print("WEEKLY STATISTICS SUMMARY")
print(f"{'='*70}")

for week_idx, start_idx in enumerate(random_week_starts):
    end_idx = start_idx + days_per_week
    week_dates_subset = test_dates[start_idx:end_idx]
    
    week_mean = ensemble_mean_denorm[start_idx:end_idx]
    week_std = ensemble_std_denorm[start_idx:end_idx]
    week_actual = actual_values_denorm[start_idx:end_idx]
    
    week_mae = mean_absolute_error(week_actual, week_mean)
    week_rmse = np.sqrt(mean_squared_error(week_actual, week_mean))
    week_mean_uncertainty = np.mean(week_std)
    
    week_start_str = pd.to_datetime(week_dates_subset[0]).strftime('%Y-%m-%d')
    week_end_str = pd.to_datetime(week_dates_subset[-1]).strftime('%Y-%m-%d')
    
    print(f"\nWeek {week_idx + 1} ({week_start_str} to {week_end_str}):")
    print(f"  MAE:         {week_mae:.4f} Mkm²")
    print(f"  RMSE:        {week_rmse:.4f} Mkm²")
    print(f"  Uncertainty: {week_mean_uncertainty:.4f} Mkm² (mean std)")
    print(f"  Mean Extent: {week_actual.mean():.2f} Mkm² (actual)")
    print(f"               {week_mean.mean():.2f} Mkm² (predicted)")